# When can we trust BIOMASS forest height?
## A three-site experiment in Gabon: Rabi, Mabounie and Lopé

**ESA MAAP Hackathon 2026 — Challenge #5 "Canopy Clash"** · main notebook of this folder

A first single-site pilot at Lopé (not included here) found three problems with the BIOMASS Forest Height (FH) product at Lopé:
1. **savanna** (3 m tall in reality) is mapped as 20 m tall forest;
2. on **steep slopes** the height is underestimated by up to 11 m;
3. two acquisitions **three weeks apart** disagree by 8 m RMSE.

But Lopé mixes savanna, hills and forest, so we could not tell these effects apart. Here we design an experiment that separates them, and we first explain **how the algorithm works**, so that each problem becomes a **testable prediction**.

| Site | Landscape | What it isolates |
|---|---|---|
| **Rabi** (south-west Gabon) | flat, closed forest, no savanna | **the control**: how good is BIOMASS in ideal conditions? |
| **Mabounie** (central Gabon) | forest with moderate hills, seen from **two directions** (ascending + descending orbits) | **terrain**: does the error follow the radar geometry? |
| **Lopé** (central Gabon) | forest + savanna + the steepest hills, also seen from both directions | **short vegetation**, steep terrain, everything combined |

For every site: all BIOMASS FH scenes of 2026, airborne LiDAR (**LVIS**, NASA AfriSAR 2016) as the main reference, spaceborne LiDAR (**GEDI** 2019–2025), land cover (**ESA WorldCover**) and terrain (**Copernicus DEM**). Statistics are always **BIOMASS − reference**: negative means BIOMASS is too low.

## 1 — How does BIOMASS measure forest height?

### The instrument
BIOMASS is a **P-band SAR** (wavelength ≈ 70 cm, the first in space). Short waves (X-, C-band) bounce off the leaves at the top of the canopy. P-band goes through leaves and twigs and interacts with **trunks, big branches and the ground**, so a single radar pixel contains echoes from the whole vertical column of the forest.

### From two images to a height: interferometric coherence
BIOMASS images the same forest several times from slightly different positions (a few days apart, same orbit track). Comparing two images pixel by pixel gives the **interferometric coherence** γ, a complex number:
- its **phase** tells *at what height* the "centre" of the echoes is;
- its **magnitude** |γ| (between 0 and 1) tells *how spread out vertically* the echoes are.

If all echoes come from one height (bare ground), |γ| = 1. If they come from a 40 m thick canopy, the different heights interfere and |γ| drops. The conversion between height and phase is the **vertical wavenumber** $k_z$ [rad/m], fixed by the viewing geometry:

$$k_z \;\approx\; \frac{4\pi}{\lambda}\,\frac{\Delta\theta}{\sin\theta}$$

where Δθ is the small difference in look angle between the two passes and θ the incidence angle.

### The model: Random Volume over Ground (RVoG)
The L2 processor inverts a **model** of the forest: a layer of scatterers of height *h* with a standard vertical profile *f(z)* over a flat ground. For the volume the model predicts

$$\gamma_{vol}(h) \;=\; \frac{\int_0^{h} f(z)\, e^{i k_z z}\,dz}{\int_0^{h} f(z)\,dz}$$

and the processor searches the height *h* (and ground-to-volume ratio) that best reproduces the measured coherences, using **3 acquisitions** (interferometric phase) or **7** (tomographic phase), all three polarisations (HH, HV, VV), a terrain model for the ground phase, and an external forest/non-forest mask. The output is **upper canopy height** (H100) at **200 m** resolution, plus a **quality layer** described by ESA as *"a percentage bias value … indicating the inversion performance"*. The mission requirement is an error below **30% for forests taller than 10 m**; ESA lists **steep terrain** and **heavy rain/flooding** as known issues of the first product version.

### Why it can fail: three predictions
The key point is that **a loss of coherence that the model cannot attribute to something else is read as forest volume**. Anything that lowers coherence or changes $k_z$ without being in the model turns into a height error:

1. **Low backscatter → noise → fake height.** Grass and bare soil return very little P-band energy. Thermal noise then lowers |γ|, and the model reads "low coherence" as "tall canopy". *Prediction: short vegetation is overestimated, and the quality indicator flags it.*
2. **Slopes change the geometry.** On a hillside the *local* incidence angle (LIA) is different from θ: smaller on slopes facing the radar, larger on slopes facing away. If the inversion uses the flat-terrain $k_z$, the height is scaled by roughly $\sin\theta / \sin(\text{LIA})$. *Prediction: the error depends on the LIA, i.e. on which side of the hill the radar sees, so the same hill should look different from ascending and descending orbits.* (If the processor already corrects $k_z$ for terrain, this effect should vanish: the data will tell.)
3. **Changes between acquisitions.** Rain, wind, wet soil or ionospheric residuals between the passes also decorrelate the signal. *Prediction: two dates disagree even if the forest did not change.*

The figure below simulates these mechanisms with the simplest version of the model (uniform vertical profile). The $k_z$ values are **illustrative**, not the operational ones.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

def vol_coherence(h, kz):
    """|gamma_vol| for a uniform vertical profile of height h (RVoG, no ground)."""
    return np.abs(np.sinc(kz * np.asarray(h) / (2 * np.pi)))

def invert_height(gamma, kz, hmax=80):
    """Height that the no-noise uniform-volume model assigns to an observed coherence magnitude."""
    hh = np.linspace(0, min(hmax, 0.999 * 2 * np.pi / kz), 4001)   # stay before the first zero of the sinc
    g = vol_coherence(hh, kz)
    return np.interp(np.asarray(gamma), g[::-1], hh[::-1])

C1, C2 = "#2a78d6", "#eb6834"
fig, ax = plt.subplots(1, 3, figsize=(16, 4.3))
h = np.linspace(0, 60, 400)
for kz, c in [(0.05, C1), (0.10, C2)]:
    ax[0].plot(h, vol_coherence(h, kz), color=c, lw=2, label=f"$k_z$ = {kz} rad/m")
ax[0].set(title="A · Taller forest → lower coherence", xlabel="Forest height h [m]", ylabel="Volume coherence |γ|")
ax[0].legend(frameon=False)

snr_db = np.linspace(-5, 20, 200)
snr = 10 ** (snr_db / 10)
for kz, c in [(0.05, C1), (0.10, C2)]:
    observed = vol_coherence(3.0, kz) * snr / (1 + snr)          # 3 m tall grass + thermal noise
    ax[1].plot(snr_db, invert_height(observed, kz), color=c, lw=2, label=f"$k_z$ = {kz} rad/m")
ax[1].axhline(3, color="#898781", ls="--", lw=1); ax[1].text(19.5, 4.5, "true height 3 m", ha="right", fontsize=8, color="#52514e")
ax[1].set(title="B · Noise is read as canopy (savanna)", xlabel="Signal-to-noise ratio [dB]", ylabel="Height estimated by the model [m]", ylim=(0, 60))

theta = 29.0
range_slope = np.linspace(-25, 25, 200)      # >0: slope facing away from the radar
lia = theta + range_slope
ax[2].plot(range_slope, 35 * (np.sin(np.radians(theta)) / np.sin(np.radians(lia)) - 1), color=C1, lw=2)
ax[2].axhline(0, color="#c3c2b7", lw=1)
ax[2].set(title="C · Slopes change $k_z$ (35 m forest, θ = 29°)", xlabel="Slope towards ← radar | away from radar → [°]",
          ylabel="Height error if flat $k_z$ is used [m]", ylim=(-25, 60))
for a in ax:
    a.spines[["top", "right"]].set_visible(False)
fig.tight_layout()
SIM_FIG = fig
plt.show()

**Reading the simulation.**
- **A:** coherence falls as the forest gets taller. That is the signal BIOMASS uses.
- **B:** a 3 m savanna with a signal-to-noise ratio (SNR) of about 10 dB is "seen" as a 15–30 m forest, the same order as the 3 m → 21 m we measured at Lopé. Only with a strong signal (> 20 dB) does the model recover the true 3 m.
- **C:** with the same forest, a slope facing *away* from the radar gives a lower height and a slope facing *towards* it a higher one. The sign is opposite on the two sides of a hill. This is the prediction we test at Mabounie with two viewing directions.

## 2 — Data

All data are real. Everything is cached under `data/<site>/`: the first run downloads, later runs read from disk.

| Dataset | What | Access |
|---|---|---|
| BIOMASS L2A FH + quality | all 2026 scenes intersecting each site, cropped to the site | ESA MAAP catalogue (**MAAP secret**, typed once, kept in memory only) |
| LVIS L3 RH98 25 m (2016) | airborne LiDAR canopy height, AfriSAR (DOI 10.3334/ORNLDAAC/1775) | NASA Earthdata (**Earthdata login**, typed once) |
| GEDI L2A RH98 (2019–2025) | spaceborne LiDAR, quality_flag = 1, degrade_flag = 0, sensitivity ≥ 0.95 | public gediDB (GFZ), no login |
| ESA WorldCover 2021, Copernicus DEM | land cover fractions, elevation, slope, aspect | public cloud-optimised GeoTIFFs |

**Passes and frames.** BIOMASS delivers each satellite pass in several adjacent *frames*. We merge the frames of the same pass (same date, track and direction, so there are no artificial seams) and keep only passes that cover at least 20% of the LiDAR survey. Small slivers would only add noise and "postage-stamp" maps.

**Viewing geometry.** For each scene we take the orbit direction (ascending/descending) and look side (left) from the catalogue, compute the satellite heading from the scene footprint, and model the incidence angle across the swath from **26.5° (near range) to 31.4° (far range)**. These values are read from the incidence-angle lookup table of a real BIOMASS L1 tomographic product. Combined with the DEM slope and aspect, this gives the **local incidence angle (LIA)** of every pixel in every scene.

In [ ]:
# Setup
import importlib.util, subprocess, sys, json, os, getpass
from pathlib import Path
for pkg, mod in [("python-pptx", "pptx"), ("earthaccess", "earthaccess")]:
    if importlib.util.find_spec(mod) is None:
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", pkg], check=True)

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.colors import LinearSegmentedColormap, ListedColormap, BoundaryNorm, Normalize, TwoSlopeNorm
from matplotlib.patches import Patch, Rectangle
from IPython.display import display, Markdown

ROOT = next((c for c in [Path.cwd().resolve(), *Path.cwd().resolve().parents]
             if (c / "src" / "__init__.py").exists() and (c / "config" / "sites.json").exists()), None)
if ROOT is None:
    raise FileNotFoundError("Open this notebook from inside the topic folder (it needs src/ and config/).")
sys.path.insert(0, str(ROOT))
from src.load.biomass import find_scenes, missing_scenes, download_scenes
from src.load.lidar import find_lvis, download_lvis
from src.load.context import COVER_LABELS, hillshade
from src.process.site import load_site
from src.process.strip import strip_frame, best_by_orbit
from src.evaluate import accuracy as acc

CFG = json.loads((ROOT / "config" / "sites.json").read_text())
SITES = CFG["sites"]
FIG = ROOT / "results" / "multisite"; FIG.mkdir(parents=True, exist_ok=True)
C_BIOMASS, C_LVIS, C_GEDI = "#2a78d6", "#eb6834", "#1baf7a"
C_ASC, C_DESC = "#6250d6", "#eda100"
HEIGHT_CMAP = LinearSegmentedColormap.from_list("height", ["#f4f1e6", "#9fcf8f", "#2e8b3e", "#0b3d1f"])
DIFF_CMAP = LinearSegmentedColormap.from_list("diff", ["#1c5cab", "#86b6ef", "#f0efec", "#f3a77f", "#b8461b"])
plt.rcParams.update({"figure.dpi": 110, "axes.spines.top": False, "axes.spines.right": False, "font.size": 10,
                     "axes.titlesize": 11, "axes.titleweight": "bold"})
FACTS, FIGS = {}, {}

def savefig(fig, name):
    p = FIG / f"{name}.png"; fig.savefig(p, dpi=150, bbox_inches="tight", facecolor="white"); FIGS[name] = p; return p
if "SIM_FIG" in globals():
    savefig(SIM_FIG, "00_mechanisms")
print("Project:", ROOT, "| sites:", ", ".join(s["name"] for s in SITES.values()))

In [ ]:
# Which BIOMASS scenes exist for each site?
SCENES = {}
for key, s in SITES.items():
    SCENES[key] = find_scenes(s["bbox"], CFG["biomass_datetime"], CFG["catalog_url"])
    sc = SCENES[key]
    todo = missing_scenes(sc, ROOT / "data" / key)
    print(f"{s['name']:9s}: {len(sc)} scenes "
          f"({', '.join(sorted({f'T{t:03d} {o[:4]}' for t, o in zip(sc.track, sc.orbit)}))}) — {len(todo)} to download")

In [ ]:
# Downloads (only what is missing). You will be asked ONCE for each secret; nothing is written to disk.
SKIP = bool(os.environ.get("SKIP_DOWNLOADS"))      # only for automated test runs without passwords
need_biomass = (not SKIP) and any(len(missing_scenes(SCENES[k], ROOT / "data" / k)) for k in SITES)
if need_biomass and not os.environ.get("ESA_CLIENT_SECRET"):
    os.environ["ESA_CLIENT_SECRET"] = getpass.getpass("ESA MAAP CLIENT_SECRET (hidden): ")
for key, s in SITES.items():
    if need_biomass and len(missing_scenes(SCENES[key], ROOT / "data" / key)):
        print(f"BIOMASS {s['name']}:")
        download_scenes(SCENES[key], s["bbox"], ROOT / "data" / key)

for key, s in SITES.items():
    site_dir = ROOT / "data" / key
    site_dir.mkdir(parents=True, exist_ok=True)
    if find_lvis(site_dir, s["lvis"]) is None and not SKIP:
        print(f"LVIS {s['name']}: downloading {s['lvis']} (Earthdata login)")
        download_lvis(CFG["lvis_base_url"] + s["lvis"], site_dir)
    print(f"LVIS {s['name']}: {'ready' if find_lvis(site_dir, s['lvis']) else 'MISSING'}")

# GEDI shots: public gediDB, no login (same query as scripts/fetch_gedi.py)
from src.load.gedi import fetch_gedi
for key, s in SITES.items():
    gp = ROOT / "data" / key / f"gedi_{key}.parquet"
    if not gp.exists() and not SKIP:
        print(f"GEDI {s['name']}: querying gediDB {CFG['gedi_start']} to {CFG['gedi_end']}")
        fetch_gedi(s["bbox"], gp, CFG["gedi_start"], CFG["gedi_end"], sensitivity_min=CFG["gedi_sensitivity_min"])
    print(f"GEDI {s['name']}: {'ready' if gp.exists() else 'MISSING'}")

In [ ]:
# Put every dataset of every site on one common grid per site
D = {key: load_site(key, s, ROOT, swath_incidence=tuple(CFG["swath_incidence_deg"]),
                    min_coverage=CFG["lvis_min_coverage_fraction"], min_pass_coverage=CFG.get("min_pass_lidar_coverage", 0.2)) for key, s in SITES.items()}
ALL = list(D.values())

## 3 — The three sites

Each site box is the extent of the NASA LVIS airborne survey. For each site we describe the landscape with *independent* data: land cover from WorldCover, elevation and slope from the Copernicus DEM.

In [ ]:
cover_cmap = ListedColormap(["#2e8b3e", "#b7d36b", "#eda100", "#c3c2b7"])
fig, ax = plt.subplots(2, len(ALL), figsize=(5.2 * len(ALL), 9))
rows = []
for j, S in enumerate(ALL):
    g, ctx = S["grid"], S["ctx"]
    ax[0, j].imshow(S["cover"], extent=g.extent, cmap=cover_cmap, norm=BoundaryNorm([-.5, .5, 1.5, 2.5, 3.5], 4), interpolation="nearest")
    if S["lvis"] is not None:
        ax[0, j].contour(g.lon, g.lat, np.isfinite(S["lvis"]).astype(float), levels=[.5], colors="k", linewidths=.6)
    w = S["win_extent"]
    ax[0, j].add_patch(Rectangle((w[0], w[2]), w[1] - w[0], w[3] - w[2], fill=False, ec="#d03b3b", lw=1.5))
    ax[0, j].set_title(f"{S['name']} — land cover (black: LVIS, red: map window)")
    ax[1, j].imshow(hillshade(ctx["dem"]), extent=g.extent, cmap="Greys_r")
    im = ax[1, j].imshow(ctx["dem"], extent=g.extent, cmap="terrain", alpha=.5)
    ax[1, j].set_title(f"{S['name']} — terrain")
    fig.colorbar(im, ax=ax[1, j], shrink=.7, label="Elevation [m]")
    v = np.isfinite(S["cover"])
    rows.append({"site": S["name"], "role": S["role"],
                 "forest %": 100 * np.mean(S["cover"][v] == 0), "savanna %": 100 * np.mean(S["cover"][v] == 2),
                 "elevation [m]": f"{max(np.nanmin(ctx['dem']), 0):.0f}–{np.nanmax(ctx['dem']):.0f}",
                 "median slope [°]": np.nanmedian(ctx["slope"]), "forest slope ≥ 15° [%]": 100 * np.nanmean(ctx["slope"][S['cover'] == 0] >= 15),
                 "BIOMASS scenes": len(S["scenes"]), "orbits": ", ".join(sorted({s['orbit'] for s in S['scenes']})),
                 "LVIS pixels": int(np.isfinite(S["lvis"]).sum()) if S["lvis"] is not None else 0,
                 "GEDI shots": 0 if S["gedi_shots"] is None else len(S["gedi_shots"])})
ax[0, 0].legend(handles=[Patch(color=c, label=l) for c, l in zip(cover_cmap.colors, COVER_LABELS)], loc="lower left", fontsize=8)
for a in ax.flat:
    a.set_xlabel("Longitude [°]"); a.set_ylabel("Latitude [°]")
fig.tight_layout(); savefig(fig, "01_sites"); plt.show()
SITE_TABLE = pd.DataFrame(rows).round(1)
display(SITE_TABLE)
FACTS["site_table"] = SITE_TABLE

In [ ]:
# All passes (frames merged): which ones are kept for the analysis?
sc_rows = [{"site": S["name"], "date": p["date"].date(), "orbit": p["orbit"], "track": p["track"],
            "frames": "+".join(str(f) for f in p["frames"]), "LiDAR area covered [%]": round(100 * p["lidar_coverage"]),
            "used": "yes" if p in S["scenes"] else "no (too small)", "best (maps)": "★" if p is S["best"] else "",
            "heading [°]": round(p["heading"], 1), "look azimuth [°]": round(p["look_az"], 1)}
           for S in ALL for p in S["all_passes"]]
display(pd.DataFrame(sc_rows))

## 4 — First look: the same forests seen by three sensors

Same colour scale everywhere, zoomed on the area where LiDAR and BIOMASS overlap (grey = no data). BIOMASS shows the pass that covers most of the LiDAR survey. LVIS is the 2016 airborne survey averaged into the BIOMASS pixels. GEDI shows the median of the shots falling in each pixel.

In [ ]:
fig, ax = plt.subplots(len(ALL), 3, figsize=(15, 4.6 * len(ALL)), squeeze=False)
norm = Normalize(0, 50)
for i, S in enumerate(ALL):
    win, ext = S["win"], S["win_extent"]
    best = S["best"]
    for j, (arr, title) in enumerate([(best["h"] if best else None, f"BIOMASS FH ({best['label']})" if best else "BIOMASS: no pass"),
                                      (S["lvis"], "LVIS RH98 2016 (flight strips)"), (None, "GEDI RH98 2019–2025 (shots)")]):
        a = ax[i, j]; a.set_facecolor("#ecebe6")
        if arr is not None:
            im = a.imshow(arr[win], extent=ext, cmap=HEIGHT_CMAP, norm=norm, interpolation="nearest")
        elif j == 2 and S["gedi_shots"] is not None:
            g = S["gedi_shots"]
            g = g[g.longitude.between(ext[0], ext[1]) & g.latitude.between(ext[2], ext[3])].sort_values("rh_98")
            a.scatter(g.longitude, g.latitude, c=g.rh_98, s=.6, cmap=HEIGHT_CMAP, norm=norm, rasterized=True)
            a.set_xlim(ext[0], ext[1]); a.set_ylim(ext[2], ext[3]); a.set_aspect("equal")
        a.set_title(f"{S['name']} — {title}", fontsize=10)
        a.set_xlabel("Longitude [°]"); a.set_ylabel("Latitude [°]")
fig.colorbar(im, ax=ax, shrink=.5, label="Height [m]")
savefig(fig, "02_first_look"); plt.show()

### 4b — Along the LiDAR strip: every sensor on the same rectangle

The airborne LiDAR was flown as **long, narrow strips**, so on a north-up map it always looks like a diagonal band in an empty box. To compare the sensors where they *all* exist, we **straighten the strip**: we find its direction (principal axis of the LVIS pixels), keep its densest band, and resample every layer along it. Each site becomes one **full rectangle** tens of kilometres long and a few kilometres wide.

From top to bottom: the LiDAR reference, then for the best ascending and best descending pass the BIOMASS height, its error against LVIS and the **local incidence angle** (brown = slope facing the radar, blue = slope facing away), and finally the terrain. The bottom panel is a **transect**: the median height across the strip every kilometre, with the GEDI shots that fall inside the strip.

Look for the same pattern at every site: **brown patches in the incidence-angle ribbon line up with blue (too low) patches in the error ribbon**.

In [ ]:
LIA_CMAP = LinearSegmentedColormap.from_list("lia", ["#5a1d06", "#f3a77f", "#f0efec", "#86b6ef", "#1c5cab"])
STRIPS = {}
for S in ALL:
    if S["lvis"] is None or not S["scenes"]:
        continue
    F = strip_frame(S); STRIPS[S["name"]] = F
    bo = best_by_orbit(S)
    rows = [("LVIS RH98 2016 (airborne LiDAR)", F["sample"](S["lvis"]), HEIGHT_CMAP, Normalize(0, 50), "Height [m]")]
    for orb in ("ascending", "descending"):
        if orb in bo:
            p = bo[orb]
            rows += [(f"BIOMASS FH — {p['label']}", F["sample"](p["h"]), HEIGHT_CMAP, Normalize(0, 50), "Height [m]"),
                     (f"BIOMASS − LVIS — {p['label']}", F["sample"](p["h"] - S["lvis"]), DIFF_CMAP, TwoSlopeNorm(0, -25, 25), "Error [m]"),
                     (f"Local incidence angle — {p['label']}", F["sample"](p["lia"]), LIA_CMAP, TwoSlopeNorm(30, 0, 60), "LIA [°]")]
    rows.append(("Terrain (Copernicus DEM)", F["sample"](S["ctx"]["dem"]), "terrain", None, "Elevation [m]"))
    rh = float(np.clip(13 * F["width_km"] / F["length_km"], .55, 1.6))
    fig = plt.figure(figsize=(14, rh * len(rows) + 3.6))
    gsp = fig.add_gridspec(len(rows) + 1, 2, width_ratios=[60, 1], height_ratios=[rh] * len(rows) + [3.0], hspace=.6, wspace=.02, top=1 - 0.45 / fig.get_figheight(), bottom=0.6 / fig.get_figheight())
    for k, (title, arr, cmap, nrm, lab) in enumerate(rows):
        a = fig.add_subplot(gsp[k, 0]); ca = fig.add_subplot(gsp[k, 1])
        im = a.imshow(arr, extent=F["extent"], cmap=cmap, norm=nrm, aspect="auto", interpolation="nearest")
        a.set_title(title, fontsize=9, loc="left"); a.set_yticks([]); a.tick_params(labelbottom=False)
        fig.colorbar(im, cax=ca); ca.tick_params(labelsize=7); ca.set_ylabel(lab, fontsize=7)
    a = fig.add_subplot(gsp[-1, 0])
    edges = np.arange(0, F["length_km"] + 1, 1.0); mids = edges[:-1] + .5
    sgrid = np.broadcast_to(F["s"], F["sample"](S["lvis"]).shape)
    idx = np.digitize(sgrid, edges) - 1
    def prof(arr):
        rib = F["sample"](arr)
        return np.array([np.nanmedian(rib[idx == k]) if np.isfinite(rib[idx == k]).sum() > 5 else np.nan for k in range(len(mids))])
    a.plot(mids, prof(S["lvis"]), color=C_LVIS, lw=2.2, label="LVIS")
    for orb, c in (("ascending", C_ASC), ("descending", C_DESC)):
        if orb in bo:
            a.plot(mids, prof(bo[orb]["h"]), color=c, lw=1.6, label=f"BIOMASS {orb}")
    if S["gedi_shots"] is not None:
        gs_s, gs_t = F["project"](S["gedi_shots"].longitude.to_numpy(), S["gedi_shots"].latitude.to_numpy())
        inb = (gs_t >= F["t_range"][0]) & (gs_t <= F["t_range"][1]) & (gs_s >= F["s_range"][0]) & (gs_s <= F["s_range"][1])
        a.scatter(gs_s[inb] - F["s_range"][0], S["gedi_shots"].rh_98.to_numpy()[inb], s=3, color=C_GEDI, alpha=.35, label="GEDI shots", rasterized=True)
    a.set(xlim=(0, F["length_km"]), ylim=(0, 65), xlabel="Distance along the LiDAR strip [km]", ylabel="Height [m]")
    a.legend(frameon=False, fontsize=8, ncol=4, loc="upper right")
    fig.suptitle(f"{S['name']}: {F['length_km']:.0f} km × {F['width_km']:.1f} km strip along the LVIS flight line (azimuth {F['azimuth']:.0f}°)",
                 fontweight="bold", y=1 - 0.12 / fig.get_figheight())
    savefig(fig, f"02b_strip_{S['key']}"); plt.show()

## 5 — Rabi, the control: how good is BIOMASS in ideal conditions?

Rabi is flat and almost entirely closed forest. No savanna, no hills: if BIOMASS has a problem here, it is the algorithm itself (or changes between dates), not the landscape.

The table summarises every site with the **median over all its scenes**, split into **landscape strata**: flat / moderate / steep forest, and savanna. The ESA requirement for comparison is **relative error < 30%** for forests taller than 10 m.

In [ ]:
ACC = acc.accuracy_table(ALL)
ACC.to_csv(FIG / "accuracy_all_scenes.csv", index=False)
SUM_L = acc.site_summary(ACC, "LVIS"); SUM_G = acc.site_summary(ACC, "GEDI")
print("Against LVIS (airborne LiDAR):"); display(SUM_L.round(2))
print("Against GEDI (spaceborne LiDAR):"); display(SUM_G.round(2))
SUM_L.to_csv(FIG / "summary_vs_lvis.csv", index=False); SUM_G.to_csv(FIG / "summary_vs_gedi.csv", index=False)
FACTS["sum_lvis"] = SUM_L

def get(df, site, stratum, col):
    r = df[(df.site == site) & (df.stratum == stratum)]
    return float(r[col].iloc[0]) if len(r) and np.isfinite(r[col].iloc[0]) else np.nan
FACTS["rabi_flat_rmse"] = get(SUM_L, SITES["rabi"]["name"], "forest, flat (<5°)", "rmse")
FACTS["rabi_flat_rel"] = get(SUM_L, SITES["rabi"]["name"], "forest, flat (<5°)", "rel_rmse")
FACTS["rabi_flat_bias"] = get(SUM_L, SITES["rabi"]["name"], "forest, flat (<5°)", "bias")

In [ ]:
# Height distributions in flat forest: BIOMASS (all dates) vs LVIS vs GEDI, per site
bins = np.arange(0, 66, 2)
fig, ax = plt.subplots(1, len(ALL), figsize=(5.2 * len(ALL), 4), sharey=True)
for a, S in zip(np.atleast_1d(ax), ALL):
    m = acc.strata(S)["forest, flat (<5°)"]
    if S["scenes"]:
        a.hist(np.concatenate([s["h"][m] for s in S["scenes"]]), bins, density=True, histtype="step", lw=2, color=C_BIOMASS, label="BIOMASS (all dates)")
    if S["lvis"] is not None:
        a.hist(S["lvis"][m], bins, density=True, histtype="step", lw=2, color=C_LVIS, label="LVIS")
    if S["gedi"] is not None:
        a.hist(S["gedi"][m], bins, density=True, histtype="step", lw=1.5, ls="--", color=C_GEDI, label="GEDI")
    a.set(title=f"{S['name']}: flat forest", xlabel="Height [m]")
np.atleast_1d(ax)[0].set_ylabel("Density"); np.atleast_1d(ax)[0].legend(frameon=False, fontsize=8)
fig.tight_layout(); savefig(fig, "03_flat_forest_histograms"); plt.show()

## 6 — Prediction 2 (terrain): does the error follow the radar geometry?

If slopes act through the viewing geometry, the error must depend on the **local incidence angle**, not on the forest. We test this in two ways, on **forest pixels only**:

1. **Error vs LIA**, separately for ascending and descending orbits, against LVIS. The dashed lines show the first-order prediction of the simple $k_z$ model (Chapter 1, panel C) for the same pixels.
2. **The same hills seen from two sides.** For pixels covered by both orbit directions we compute *height(ascending) − height(descending)*. The forest is identical, only the geometry changes. If the difference follows *LIA(ascending) − LIA(descending)*, the error is geometric.

In [ ]:
LIA_TAB = pd.concat([pd.DataFrame()] + [acc.error_vs_lia(S) for S in ALL if S["lvis"] is not None and S["scenes"]], ignore_index=True)
display(LIA_TAB.round(2)); LIA_TAB.to_csv(FIG / "error_vs_lia.csv", index=False)
sites_lia = [S for S in ALL if len(LIA_TAB) and S["name"] in set(LIA_TAB.site)]
if sites_lia:
    fig, ax = plt.subplots(1, len(sites_lia), figsize=(5.2 * len(sites_lia), 4.2), sharey=True, squeeze=False)
    for a, S in zip(ax[0], sites_lia):
        t = LIA_TAB[LIA_TAB.site == S["name"]]
        for orb, c in [("ascending", C_ASC), ("descending", C_DESC)]:
            u = t[t.orbit == orb]
            if len(u):
                a.plot(u.lia_mid, u.bias, "o-", color=c, lw=2, ms=6, label=f"{orb} (data)")
                a.plot(u.lia_mid, u.kz_prediction, "--", color=c, lw=1.2, label=f"{orb} ($k_z$ model)")
        a.axhline(0, color="#c3c2b7", lw=1)
        a.set(title=f"{S['name']}: forest error vs LIA", xlabel="Local incidence angle [°]")
    ax[0, 0].set_ylabel("BIOMASS − LVIS bias [m]"); ax[0, 0].legend(frameon=False, fontsize=8)
    fig.tight_layout(); savefig(fig, "04_error_vs_lia"); plt.show()

**How to read it.** Each line is the forest error in bins of local incidence angle (LIA). Around the mean incidence (≈ 27–32°) the terrain is effectively flat for the radar.

- If the **$k_z$ hypothesis** of Chapter 1 were the main effect, slopes **facing the radar** (small LIA) would be *over*estimated and slopes facing away (large LIA) *under*estimated: the dashed lines.
- The alternative is **foreshortening**. A slope facing a side-looking radar is compressed in slant range: one radar pixel collects the ground and trees of a much longer stretch of hillside, so the **ground contribution grows** and the echoes come from a wider range of terrain heights. The inversion then places the phase centre lower and loses the canopy top, which gives an *under*estimation that is strongest at small LIA. Slopes facing away are stretched instead, and suffer much less.

So the **sign of the error at small LIA** tells the two mechanisms apart. The conclusions (Chapter 10) state which one the data support for every site and orbit.

In [ ]:
AD = {}
for S in ALL:
    tab, r, n = acc.asc_desc_difference(S)
    if len(tab):
        AD[S["name"]] = (tab, r, n)
        print(f"{S['name']}: {n:,} forest pixels seen from both directions, correlation(Δh, ΔLIA) = {r:.2f}")
        display(tab.round(2))
if AD:
    fig, ax = plt.subplots(1, len(AD), figsize=(5.2 * len(AD), 4.2), sharey=True, squeeze=False)
    for a, (name, (tab, r, n)) in zip(ax[0], AD.items()):
        a.bar(tab.dlia_mid, tab.dh_median, width=3, color=C_BIOMASS)
        a.axhline(0, color="#c3c2b7", lw=1)
        a.set(title=f"{name}: same forest, two viewing directions", xlabel="LIA ascending − LIA descending [°]")
        a.text(.02, .95, f"r = {r:.2f}, n = {n:,}", transform=a.transAxes, va="top", fontsize=9, color="#52514e")
    ax[0, 0].set_ylabel("Height ascending − descending [m]")
    fig.tight_layout(); savefig(fig, "05_asc_desc"); plt.show()
    FACTS["asc_desc"] = {k: (v[1], v[2]) for k, v in AD.items()}
else:
    print("No site has both ascending and descending scenes with valid overlap yet.")

## 7 — Prediction 1 (short vegetation): savanna or error?

Short vegetation (< 10 m) has a low P-band signal. Chapter 1 predicts BIOMASS should **overestimate** it. *Recall* = share of truly short pixels (LVIS) that BIOMASS also calls short. *Precision* = share of BIOMASS "short" pixels that are really short. Values are medians over scenes.

In [ ]:
SV = pd.concat([pd.DataFrame()] + [acc.short_vegetation(S, CFG["low_height_m"]) for S in ALL], ignore_index=True)
display(SV.round(1)); SV.to_csv(FIG / "short_vegetation.csv", index=False)
FACTS["short"] = SV

## 8 — Prediction 3 (time): do two dates agree?

For every pair of scenes of a site we compare BIOMASS with itself, on forest pixels. **Same track** pairs have identical geometry, so any difference comes from changes between acquisitions (weather, soil moisture, ionosphere, processing). Pairs from **opposite directions** add the geometry effect of Chapter 6.

In [ ]:
REP = pd.concat([pd.DataFrame()] + [acc.repeatability(S, mask=(S["cover"] == 0)) for S in ALL if len(S["scenes"]) > 1], ignore_index=True)
if len(REP):
    display(REP[["site", "a", "b", "days", "geometry", "n", "bias", "rmse", "r"]].round(2))
    REP.to_csv(FIG / "repeatability.csv", index=False)
    rs = REP.groupby(["site", "geometry"]).agg(pairs=("rmse", "size"), median_rmse=("rmse", "median"), median_r=("r", "median")).reset_index()
    display(rs.round(2)); FACTS["rep"] = rs
    fig, ax = plt.subplots(figsize=(8, 4.2))
    for geo, c in [("same track", C_BIOMASS), ("same direction", C_GEDI), ("opposite directions", C_LVIS)]:
        u = REP[REP.geometry == geo]
        ax.scatter(u.days, u.rmse, s=50, color=c, label=geo, edgecolor="white", linewidth=1)
    ax.set(xlabel="Days between the two scenes", ylabel="RMSE between the two BIOMASS maps [m]", title="BIOMASS vs BIOMASS (forest pixels)")
    ax.legend(frameon=False); fig.tight_layout(); savefig(fig, "06_repeatability"); plt.show()
else:
    print("Fewer than two scenes per site: run the download cell first.")

## 9 — Where can BIOMASS forest height be trusted?

One picture for all sites: the relative error against LVIS in each landscape stratum (median over scenes). The dashed line is the mission requirement (30%, for forests taller than 10 m).

In [ ]:
order = ["forest, flat (<5°)", "forest, moderate (5–15°)", "forest, steep (≥15°)", "savanna"]
cols = [C_BIOMASS, C_GEDI, C_LVIS, "#e87ba4"]
t = SUM_L[SUM_L.stratum.isin(order)]
if len(t):
    names = [S["name"] for S in ALL]
    fig, ax = plt.subplots(figsize=(10, 4.5))
    w = .8 / len(order)
    for k, (st, c) in enumerate(zip(order, cols)):
        vals = [get(SUM_L, n, st, "rel_rmse") for n in names]
        ax.bar(np.arange(len(names)) + (k - 1.5) * w, vals, width=w * .92, color=c, label=st)
    ax.axhline(30, color="#52514e", ls="--", lw=1); ax.text(len(names) - .55, 31, "requirement 30%", fontsize=8, color="#52514e", ha="right")
    ax.set_xticks(range(len(names)), names); ax.set_ylabel("Relative RMSE vs LVIS [%]")
    ax.set_title("BIOMASS FH error by landscape"); ax.legend(frameon=False, fontsize=8, ncol=2)
    fig.tight_layout(); savefig(fig, "07_where_to_trust"); plt.show()

## 10 — Conclusions

The key numbers below are generated from the tables above, so they always match the data.

In [ ]:
L = []
f = FACTS
if np.isfinite(f.get("rabi_flat_rmse", np.nan)):
    L.append(f"Control (Rabi, flat forest): RMSE {f['rabi_flat_rmse']:.1f} m ({f['rabi_flat_rel']:.0f}% of mean height), "
             f"bias {f['rabi_flat_bias']:+.1f} m against airborne LiDAR — {'within' if f['rabi_flat_rel'] < 30 else 'outside'} the 30% requirement.")
for name, (r, n) in f.get("asc_desc", {}).items():
    L.append(f"Terrain ({name}): over {n:,} forest pixels seen from both orbits, the height difference follows the LIA difference with r = {r:.2f}"
             f" — {'the slope error is mainly geometric' if abs(r) >= 0.3 else 'geometry explains only a small part of the slope error'}.")
if len(LIA_TAB):
    for (site, orb), t in LIA_TAB.groupby(["site", "orbit"]):
        t = t.sort_values("lia_mid")
        low = t[t.lia_mid < 20]; mid = t[(t.lia_mid >= 25) & (t.lia_mid < 35)]
        if len(low) and len(mid):
            b_low = np.average(low.bias, weights=low.n); b_mid = np.average(mid.bias, weights=mid.n)
            if b_low < b_mid - 3:
                verdict = "slopes facing the radar are strongly UNDERestimated, opposite to the simple k_z model: foreshortening dominates"
            elif b_low > b_mid + 3:
                verdict = "slopes facing the radar are OVERestimated, as predicted by the k_z model"
            else:
                verdict = "no clear dependence on the viewing geometry"
            L.append(f"Terrain ({site}, {orb}): forest bias {b_low:+.1f} m on slopes facing the radar (LIA < 20°) "
                     f"vs {b_mid:+.1f} m on radar-flat terrain (LIA 25–35°): {verdict}.")
if "short" in f and len(f["short"]):
    s = f["short"][f["short"].cover == "savanna"]
    for _, row in s.iterrows():
        L.append(f"Short vegetation ({row.site}): LVIS {row.lvis_median:.0f} m, BIOMASS {row.biomass_median:.0f} m; BIOMASS finds {row.recall:.0f}% of truly short pixels.")
if "rep" in f:
    for _, row in f["rep"].iterrows():
        L.append(f"Time ({row.site}, {row.geometry}): BIOMASS vs BIOMASS RMSE {row.median_rmse:.1f} m (median of {row.pairs} pairs).")
FACTS["conclusions"] = L
display(Markdown("\n".join(f"{i}. {l}" for i, l in enumerate(L, 1)) if L else "_Run the download cell to generate the conclusions._"))

**Limitations.** LVIS is from 2016 and GEDI from 2019–2025, while BIOMASS is from 2026: logging or regrowth add noise, so we always look at medians and at strata. The incidence angle is modelled across the swath from a real BIOMASS lookup table, not read from each product. The local incidence uses a 100 m DEM, while radar layover and shadow happen at finer scale. WorldCover is itself a model. The $k_z$ simulation is a first-order illustration, not the operational processor.

## 11 — Automatic report

The cell below rebuilds the slide deck from the figures and numbers of this notebook.

In [ ]:
from pptx import Presentation
from pptx.util import Inches, Pt
from pptx.dml.color import RGBColor
from PIL import Image

INK, MUTED, ACCENT = RGBColor(0x0B, 0x0B, 0x0B), RGBColor(0x52, 0x51, 0x4E), RGBColor(0x2E, 0x8B, 0x3E)
prs = Presentation(); prs.slide_width, prs.slide_height = Inches(13.333), Inches(7.5); BLANK = prs.slide_layouts[6]

def textbox(slide, x, y, w, h, paras, size=16, color=INK, bold_first=False):
    tf = slide.shapes.add_textbox(Inches(x), Inches(y), Inches(w), Inches(h)).text_frame; tf.word_wrap = True
    for i, t in enumerate(paras):
        p = tf.paragraphs[0] if i == 0 else tf.add_paragraph()
        p.text = t; p.font.size = Pt(size); p.font.color.rgb = color; p.font.bold = bold_first and i == 0; p.space_after = Pt(8)

def header(slide, title, kicker=None):
    bar = slide.shapes.add_shape(1, 0, 0, prs.slide_width, Inches(0.12)); bar.fill.solid(); bar.fill.fore_color.rgb = ACCENT; bar.line.fill.background()
    textbox(slide, 0.5, 0.3, 12.3, 0.8, [title], size=28, bold_first=True)
    if kicker: textbox(slide, 0.5, 0.95, 12.3, 0.5, [kicker], size=14, color=MUTED)

def picture(slide, path, x, y, mw, mh):
    w, h = Image.open(path).size; s = min(mw / w, mh / h)
    slide.shapes.add_picture(str(path), Inches(x + (mw - w * s) / 2), Inches(y + (mh - h * s) / 2), Inches(w * s), Inches(h * s))

def fig_slide(title, kicker, key, bullets, full=False):
    if key not in FIGS: return
    s = prs.slides.add_slide(BLANK); header(s, title, kicker)
    if full:
        picture(s, FIGS[key], 0.4, 1.55, 12.5, 4.6); textbox(s, 0.6, 6.25, 12, 1.1, bullets, size=14)
    else:
        picture(s, FIGS[key], 0.4, 1.55, 8.6, 5.6); textbox(s, 9.2, 1.7, 3.8, 5.4, bullets, size=15)

s = prs.slides.add_slide(BLANK)
bg = s.shapes.add_shape(1, 0, 0, prs.slide_width, prs.slide_height); bg.fill.solid(); bg.fill.fore_color.rgb = RGBColor(0x0B, 0x3D, 0x1F); bg.line.fill.background()
textbox(s, .8, 2.2, 11.5, 1.2, ["When can we trust BIOMASS forest height?"], size=40, color=RGBColor(255, 255, 255), bold_first=True)
textbox(s, .8, 3.5, 11.5, 1.5, ["A three-site experiment in Gabon: Rabi (flat forest), Mabounie (hilly forest), Lopé (forest–savanna)",
                                "ESA MAAP Hackathon 2026 — Challenge #5 — generated automatically by the notebook"], size=18, color=RGBColor(0xDD, 0xE8, 0xDD))
s = prs.slides.add_slide(BLANK); header(s, "How BIOMASS measures height", "P-band PolInSAR inversion of a Random Volume over Ground model")
textbox(s, .6, 1.7, 12, 5.5, [
    "Several passes of the same track → interferometric coherence: phase = height of the echoes, magnitude = their vertical spread",
    "The processor fits a forest model (volume of height h over flat ground) to the coherences; 3 (INT) or 7 (TOM) acquisitions, HH/HV/VV",
    "Output: upper canopy height (H100), 200 m, with a quality/bias indicator; requirement < 30% error for forests > 10 m",
    "Every coherence loss is read as forest volume → three predictions:",
    "   1. low backscatter (savanna) → noise read as canopy → overestimation",
    "   2. slopes change the local incidence angle and the vertical wavenumber → error depends on viewing geometry",
    "   3. changes between passes (rain, soil moisture, ionosphere) → dates disagree"], size=17)
fig_slide("The three sites", "Each box = NASA LVIS airborne survey", "01_sites",
          [f"{r['site']}: {r['role']}, forest {r['forest %']:.0f}%, savanna {r['savanna %']:.0f}%, {r['BIOMASS scenes']} scenes" for _, r in SITE_TABLE.iterrows()], full=True)
fig_slide("Three sensors, one forest", "Same colour scale", "02_first_look", ["BIOMASS 2026 (radar)", "LVIS 2016 (airborne LiDAR)", "GEDI 2019–2025 (spaceborne LiDAR)"])
for S in ALL:
    fig_slide(f"Along the LiDAR strip: {S['name']}", "All sensors on one straightened rectangle; bottom: transect with GEDI shots",
              f"02b_strip_{S['key']}", ["Brown (slope facing the radar) ↔ blue (BIOMASS too low): foreshortening"], full=True)
fig_slide("Flat forest: the best case", "Height distributions in forest with slope < 5°", "03_flat_forest_histograms", [L[0]] if L else [], full=True)
fig_slide("Why it can fail: three mechanisms", "Simulation with a uniform-volume RVoG model (illustrative k_z)", "00_mechanisms",
          ["A: taller forest, lower coherence", "B: noise read as canopy (savanna)", "C: slopes change k_z (to be tested)"], full=True)
fig_slide("Terrain: error vs local incidence angle", "Forest only; dashed = first-order kz prediction", "04_error_vs_lia", [l for l in L if l.startswith("Terrain")][:2], full=True)
fig_slide("Same forest, two viewing directions", "Height(asc) − height(desc) vs LIA(asc) − LIA(desc)", "05_asc_desc", [l for l in L if l.startswith("Terrain")][:2], full=True)
fig_slide("Time: BIOMASS against itself", "Pairs of scenes, forest pixels", "06_repeatability", [l for l in L if l.startswith("Time")][:4])
fig_slide("Where can BIOMASS FH be trusted?", "Relative RMSE vs LVIS by landscape", "07_where_to_trust", [l for l in L if l.startswith("Short")][:2] + ["Dashed: mission requirement 30%"])
s = prs.slides.add_slide(BLANK); header(s, "Conclusions", "Generated from the notebook results")
textbox(s, .6, 1.6, 12.1, 5.7, [f"{i}. {l}" for i, l in enumerate(L, 1)] or ["Run the downloads to generate conclusions."], size=15)
report = ROOT / "results" / "Gabon_three_sites_report.pptx"; prs.save(report)
print("Report:", report, f"({len(prs.slides)} slides)")